# 02 — Data Cleaning & Merging

This notebook cleans, standardizes, and merges the three raw datasets (news headlines, crude oil prices, gold prices) into a single master dataset for analysis.

---
## Setup & Configuration



In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


BASE_DIR = Path().resolve()


if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

---
## Load Raw Data

In [32]:
gold_prices = pd.read_csv(DATA_RAW / "gold_prices.csv")
oil_prices = pd.read_csv(DATA_RAW / "crude_oil_prices.csv")

news = pd.read_csv(DATA_RAW / "abcnews-date-text.csv")

gold_prices['Date'] = pd.to_datetime(gold_prices['Date'], errors='coerce')
oil_prices['Date'] = pd.to_datetime(oil_prices['Date'], errors='coerce')

oil_prices = oil_prices.dropna(subset=['Date'])
gold_prices = gold_prices.dropna(subset=['Date'])
news = news.dropna(subset=['publish_date'])

news['publish_date'] = pd.to_datetime(
    news['publish_date'],
    format='%Y%m%d',
    errors='coerce'
)

print("Gold:", gold_prices['Date'].min(), "→", gold_prices['Date'].max())
print("Oil:", oil_prices['Date'].min(), "→", oil_prices['Date'].max())
print("News:", news['publish_date'].min(), "→", news['publish_date'].max())

Gold: 2003-01-02 00:00:00 → 2021-12-30 00:00:00
Oil: 2003-01-02 00:00:00 → 2021-12-30 00:00:00
News: 2003-02-19 00:00:00 → 2021-12-31 00:00:00


---
## Flatten & Rename Commodity Columns
The raw commodity CSVs from yfinance may have multi-level column headers. We flatten these and prefix each column with `oil_` or `gold_` to prevent naming collisions when merging.


In [ ]:
def clean_price_data(df, prefix):
    
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(1)
    
    rename_map = {
        'Open': f'{prefix}_open',
        'High': f'{prefix}_high',
        'Low': f'{prefix}_low',
        'Close': f'{prefix}_close',
        'Volume': f'{prefix}_volume'
    }
    
    df = df.rename(columns=rename_map)
    
    
    for col in df.columns:
        if col != 'Date':
            df[col] = pd.to_numeric(df[col], errors='coerce')


    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.sort_values('Date')
        df = df.set_index('Date')
    
    return df

oil_prices = clean_price_data(oil_prices, "oil")
gold_prices = clean_price_data(gold_prices, "gold")

print("Oil columns:", oil_prices.columns)
print("Gold columns:", gold_prices.columns)

print("\nOil head:")
print(oil_prices.head())

print("\nGold head:")
print(gold_prices.head())

Oil columns: Index(['oil_close', 'oil_high', 'oil_low', 'oil_open', 'oil_volume'], dtype='object')
Gold columns: Index(['gold_close', 'gold_high', 'gold_low', 'gold_open', 'gold_volume'], dtype='object')

Oil head:
            oil_close   oil_high  oil_low   oil_open  oil_volume
Date                                                            
2003-01-02  31.850000  32.090000    31.40  31.549999       62480
2003-01-03  33.080002  33.200001    31.90  32.080002       68416
2003-01-06  32.099998  32.849998    31.91  32.650002       98247
2003-01-07  31.080000  31.700001    30.51  31.549999      124279
2003-01-08  30.559999  30.700001    29.75  30.250000      108037

Gold head:
            gold_close   gold_high    gold_low   gold_open  gold_volume
Date                                                                   
2003-01-02  346.100006  346.100006  346.100006  346.100006            3
2003-01-03  351.200012  351.200012  345.200012  345.200012            0
2003-01-06  351.700012  351.70

---
## Aggregate Daily News

In [34]:
daily_news = news.groupby('publish_date').agg(
    headline_count=('headline_text', 'size'),
    combined_headlines=('headline_text', lambda x: ' | '.join(x.astype(str)))
)

print("Shape:", daily_news.shape)
print("\nSample:")
print(daily_news.head())

Shape: (6882, 2)

Sample:
              headline_count  \
publish_date                   
2003-02-19               198   
2003-02-20               250   
2003-02-21               250   
2003-02-22               126   
2003-02-23               136   

                                             combined_headlines  
publish_date                                                     
2003-02-19    aba decides against community broadcasting lic...  
2003-02-20    15 dead in rebel bombing raid philippines army...  
2003-02-21    accc too timid in petrol price investigations ...  
2003-02-22    86 confirmed dead after us nightclub fire | ac...  
2003-02-23    accused people smuggler to face darwin court |...  


---
## Handle Non-Trading Days
**The key problem:** News is published 7 days a week, but markets are open ~5 days a week (excluding holidays). If a major geopolitical event happens on Saturday, the market can't react until Monday.

**The strategy:** We need to "roll forward" weekend/holiday news to the next trading day.

In [35]:
trading_days = oil_prices.index.sort_values()
trading_days_np = trading_days.values


news_dates = daily_news.index.values

indices = np.searchsorted(trading_days_np, news_dates)
indices = np.clip(indices, 0, len(trading_days_np) - 1)


adjusted_dates = trading_days_np[indices]


daily_news_adjusted = daily_news.copy()
daily_news_adjusted['adjusted_date'] = adjusted_dates


final_news = daily_news_adjusted.groupby('adjusted_date').agg(
    headline_count=('headline_count', 'sum'),
    combined_headlines=('combined_headlines', lambda x: ' | '.join(x))
)

print("Final Shape:", final_news.shape)
print("\nSample:")
print(final_news.head(10))

Final Shape: (4737, 2)

Sample:
               headline_count  \
adjusted_date                   
2003-02-19                198   
2003-02-20                250   
2003-02-21                250   
2003-02-24                512   
2003-02-25                250   
2003-02-26                250   
2003-02-27                221   
2003-02-28                249   
2003-03-03                576   
2003-03-04                215   

                                              combined_headlines  
adjusted_date                                                     
2003-02-19     aba decides against community broadcasting lic...  
2003-02-20     15 dead in rebel bombing raid philippines army...  
2003-02-21     accc too timid in petrol price investigations ...  
2003-02-24     86 confirmed dead after us nightclub fire | ac...  
2003-02-25     4 million pay out for sacked ceo | aids organi...  
2003-02-26     abs defends nt population shrinking claim | ac...  
2003-02-27     500 million developm

---
## Merge Into Master Dataset

In [36]:
merged_prices = oil_prices.join(
    gold_prices,
    how='inner'
)

final_news.index = pd.to_datetime(final_news.index)

final_df = merged_prices.join(
    final_news,
    how='left'
)

final_df['headline_count'] = final_df['headline_count'].fillna(0)
final_df['combined_headlines'] = final_df['combined_headlines'].fillna('')


final_df = final_df.reset_index()

print("Final Shape:", final_df.shape)

print("\nColumns:")
print(final_df.columns)

print("\nSample:")
print(final_df.head())

Final Shape: (4770, 13)

Columns:
Index(['Date', 'oil_close', 'oil_high', 'oil_low', 'oil_open', 'oil_volume',
       'gold_close', 'gold_high', 'gold_low', 'gold_open', 'gold_volume',
       'headline_count', 'combined_headlines'],
      dtype='object')

Sample:
        Date  oil_close   oil_high  oil_low   oil_open  oil_volume  \
0 2003-01-02  31.850000  32.090000    31.40  31.549999       62480   
1 2003-01-03  33.080002  33.200001    31.90  32.080002       68416   
2 2003-01-06  32.099998  32.849998    31.91  32.650002       98247   
3 2003-01-07  31.080000  31.700001    30.51  31.549999      124279   
4 2003-01-08  30.559999  30.700001    29.75  30.250000      108037   

   gold_close   gold_high    gold_low   gold_open  gold_volume  \
0  346.100006  346.100006  346.100006  346.100006            3   
1  351.200012  351.200012  345.200012  345.200012            0   
2  351.700012  351.700012  351.100006  351.700012            2   
3  347.299988  349.299988  347.299988  349.299988  

---
## Save & Validate

In [37]:
final_df.to_csv(PROCESSED_DIR / "master_dataset.csv", index=False)

print("Dataset saved!")

print("\nFinal Shape:", final_df.shape)

print("\nDate Range:")
print(final_df['Date'].min(), "→", final_df['Date'].max())

print("\nColumns:")
print(final_df.columns.tolist())

print("\nNull Values:")
print(final_df.isnull().sum())


final_df['Date'] = pd.to_datetime(final_df['Date'])


final_df = final_df.sort_values('Date')

print("=" * 60)
print("MASTER DATASET SUMMARY")
print("=" * 60)
print(f"\nDate Range: {final_df['Date'].min()} → {final_df['Date'].max()}")
print(f"Total Trading Days: {len(final_df):,}")
print(f"\nOil Price Range: ${final_df['oil_close'].min():.2f} – ${final_df['oil_close'].max():.2f}")
print(f"Gold Price Range: ${final_df['gold_close'].min():.2f} – ${final_df['gold_close'].max():.2f}")
print(f"\nHeadline Count Range: {final_df['headline_count'].min():.0f} – {final_df['headline_count'].max():.0f}")
print(f"Avg Headlines/Day: {final_df['headline_count'].mean():.1f}")
print(f"Days with Zero Headlines: {(final_df['headline_count'] == 0).sum()}")
print(f"\nNull Values: {final_df.isnull().sum().sum()}")
print("=" * 60)




Dataset saved!

Final Shape: (4770, 13)

Date Range:
2003-01-02 00:00:00 → 2021-12-30 00:00:00

Columns:
['Date', 'oil_close', 'oil_high', 'oil_low', 'oil_open', 'oil_volume', 'gold_close', 'gold_high', 'gold_low', 'gold_open', 'gold_volume', 'headline_count', 'combined_headlines']

Null Values:
Date                  0
oil_close             0
oil_high              0
oil_low               0
oil_open              0
oil_volume            0
gold_close            0
gold_high             0
gold_low              0
gold_open             0
gold_volume           0
headline_count        0
combined_headlines    0
dtype: int64
MASTER DATASET SUMMARY

Date Range: 2003-01-02 00:00:00 → 2021-12-30 00:00:00
Total Trading Days: 4,770

Oil Price Range: $-37.63 – $145.29
Gold Price Range: $321.50 – $2051.50

Headline Count Range: 0 – 1112
Avg Headlines/Day: 260.4
Days with Zero Headlines: 38

Null Values: 0
